# OSL PPO Notebook — Bilateral Sensor + Larva Connectome

Colab-ready end-to-end notebook. Edit hyperparameters directly inside each cell.

Setup
- Env: `OslEnv` — bilateral sensors (0.15 mm), independent head/body rotation, obs `[c_L, c_R, prev_v, prev_body_omega, prev_head_omega]`, action `[v, body_omega, head_omega]`.
- Actor: `Connectome` (387 base nodes + 2 sensor input + 32 latent output, ~17k learnable edges, 6 message-passing steps) → concat efference copy → tanh-Gaussian.
- Critic: stateless 2-layer MLP over the 5-D obs.
- Trainer: custom on-policy PPO, separate actor/critic optimizers, sequence update via BPTT through the connectome.

Repo clone is mandatory — connectome CSVs live in `assets/connectome/`.

In [ ]:
# Colab setup. Re-running is safe.
import os, sys, subprocess

REPO_URL = 'https://github.com/InHyunseo/Brain-inspired-OSL.git'
REPO_DIR = '/content/2d-osl' if os.path.isdir('/content') else os.path.abspath('2d-osl')
if not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', 'clone', REPO_URL, REPO_DIR])
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

%pip install --quiet -r requirements.txt

for p in ('assets/connectome/weights.csv', 'assets/connectome/metadata.csv'):
    assert os.path.exists(p), f'Missing {p} — clone failed?'
print('repo:', REPO_DIR, '\ncwd :', os.getcwd())

## Smoke check

In [ ]:
import torch, numpy as np
from src.envs.osl_env import OslEnv
from src.models.policy import Policy

env = OslEnv()
obs, info = env.reset(seed=0)
print('obs', obs.shape, 'action_space', env.action_space.shape)

policy = Policy(
    weights_csv='assets/connectome/weights.csv',
    metadata_csv='assets/connectome/metadata.csv',
    latent_dim=32, message_passing_steps=6,
)
print('connectome:', policy.connectome.describe())
print('actor params :', sum(p.numel() for p in policy.actor_parameters()))
print('critic params:', sum(p.numel() for p in policy.critic_parameters()))

## Training

All knobs are plain Python variables — change them, re-run the cell. The full plan default is `[(0,0.0,1500000),(1,0.3,500000),(1,0.6,500000),(2,1.0,1000000)]` (3.5M total). The notebook ships a smaller Colab-friendly schedule so you can iterate; bump the per-phase counts when ready for a real run.

### Live TensorBoard

In [ ]:
# ===== TensorBoard (run BEFORE the training cell to watch curves live) =====
# Works in Colab and local Jupyter. Writers go to <RUN_DIR>/tb/ inside the run.
import os
os.makedirs('runs', exist_ok=True)
%load_ext tensorboard
%tensorboard --logdir runs --port 6006


In [ ]:
# ===== HYPERPARAMETERS — edit freely =====
# Curriculum: list of (noise_stage, noise_strength, env_steps).
# Stage 0 = clean Gaussian, 1 = static white noise, 2 = AR(1) plume advanced each step.
PHASES = [
    (0, 0.0, 200_000),
    (1, 0.3, 100_000),
    (1, 0.6, 100_000),
    (2, 1.0, 200_000),
]

# Env
ENV_KW = dict(
    sensor_spacing_mm=0.15,
    episode_seconds=120.0,
    arena_width_mm=80.0, arena_height_mm=120.0,
    source_x_mm=40.0, source_y_mm=100.0,
    gaussian_sigma_mm=30.0, success_radius_mm=7.5,
)

# Trainer (PPOConfig)
PPO_KW = dict(
    rollout_steps=128, num_envs=8, parallel_envs=True,
    update_epochs=4, minibatch_envs=4,
    gamma=0.99, gae_lambda=0.95, clip_epsilon=0.2,
    entropy_coef=0.005, value_loss_coef=0.5,
    actor_lr=3e-4, critic_lr=1e-3,
    actor_max_grad_norm=0.5, critic_max_grad_norm=0.5,
    log_std_init=-0.5,
    latent_dim=32, message_passing_steps=6,
    weights_csv='assets/connectome/weights.csv',
    metadata_csv='assets/connectome/metadata.csv',
    eval_interval_updates=10, eval_episodes=3,
    log_every_updates=1, checkpoint_every_timesteps=100_000,
    seed=0,
    device='auto',  # 'cpu' to force CPU
)
# ==========================================

import os, time, json, torch
from src.agents.ppo_agent import PPOConfig, PPOTrainer

RUN_DIR = os.path.join('runs', f'ppo_nb_{time.strftime("%Y%m%d_%H%M%S")}')
os.makedirs(os.path.join(RUN_DIR, 'plots'), exist_ok=True)
with open(os.path.join(RUN_DIR, 'config.json'), 'w') as f:
    json.dump({'env': ENV_KW, 'ppo': PPO_KW, 'phases': PHASES}, f, indent=2)
print('[run_dir]', RUN_DIR)

env_cfg = {**ENV_KW, 'noise_stage': 0, 'noise_strength': 0.0, 'seed': PPO_KW['seed']}
cfg = PPOConfig(**PPO_KW)

trainer = PPOTrainer(env_cfg, cfg, run_dir=RUN_DIR)
summary = {}
try:
    for stage, strength, steps in PHASES:
        print(f'\n=== Phase noise_stage={stage} strength={strength} steps={steps} ===')
        trainer.runner.set_noise_stage(stage, strength)
        summary = trainer.train(phase_timesteps=steps)
    trainer.save_final(summary)
    print('\n[final summary]\n' + json.dumps(summary, indent=2))
finally:
    trainer.close()

## Training curves

In [ ]:
import json, os
import matplotlib.pyplot as plt

log_path = os.path.join(RUN_DIR, 'training_log.jsonl')
rows = [json.loads(l) for l in open(log_path)]
steps = [r['total_steps'] for r in rows]

fig, ax = plt.subplots(2, 2, figsize=(12, 7))
ax[0,0].plot(steps, [r['recent_return_mean'] for r in rows]);   ax[0,0].set_title('recent return'); ax[0,0].grid(alpha=.3)
ax[0,1].plot(steps, [r['recent_success_rate'] for r in rows]);  ax[0,1].set_title('success rate'); ax[0,1].grid(alpha=.3)
ax[1,0].plot(steps, [r['actor_loss'] for r in rows], label='actor')
ax[1,0].plot(steps, [r['critic_loss'] for r in rows], label='critic'); ax[1,0].legend(); ax[1,0].set_title('losses'); ax[1,0].grid(alpha=.3)
ax[1,1].plot(steps, [r['entropy'] for r in rows]);               ax[1,1].set_title('entropy'); ax[1,1].grid(alpha=.3)
for a in ax.flat: a.set_xlabel('env steps')
fig.tight_layout(); plt.show()

## Evaluation + GIF
White star markers in the GIF mark frames where `event_is_high_cast_like` fired — coarse proxy for emergent stop-and-cast.

In [ ]:
# ===== EVAL HYPERPARAMETERS =====
EVAL_NOISE_STAGE = 2
EVAL_NOISE_STRENGTH = 1.0
EVAL_EPISODES = 20
EVAL_SEED_BASE = 20_000
# ================================

import torch, numpy as np, os
from IPython.display import Image as DisplayImage, display
from src.envs.osl_env import EnvConfig, OslEnv
from src.agents.ppo_agent import PPOConfig
from src.models.policy import Policy
from src.utils.plotter import render_rollout_frame, save_gif

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt = torch.load(os.path.join(RUN_DIR, 'ckpt_final.pt'), map_location=device, weights_only=False)
cfg = PPOConfig(**ckpt['agent_config'])
policy = Policy(
    weights_csv=cfg.weights_csv, metadata_csv=cfg.metadata_csv,
    latent_dim=cfg.latent_dim, message_passing_steps=cfg.message_passing_steps,
    log_std_init=cfg.log_std_init,
).to(device)
policy.load_state_dict(ckpt['policy_state_dict']); policy.eval()

def make_eval_env(seed):
    cfg_dict = {**ENV_KW, 'noise_stage': EVAL_NOISE_STAGE, 'noise_strength': EVAL_NOISE_STRENGTH, 'seed': seed}
    return OslEnv(EnvConfig.from_dict(cfg_dict))

rets, succ, best = [], [], (-float('inf'), None)
for i in range(EVAL_EPISODES):
    seed = EVAL_SEED_BASE + i
    env = make_eval_env(seed)
    obs, _ = env.reset(seed=seed)
    actor_state, critic_state = policy.initial_states(1, device)
    mask = torch.zeros(1, 1, device=device)
    ret, success = 0.0, False
    while True:
        obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            a, _, _, n_as, n_cs = policy.act(obs_t, actor_state, critic_state, mask, deterministic=True)
        obs, r, term, trunc, info = env.step(a.squeeze(0).cpu().numpy())
        ret += float(r)
        done = bool(term or trunc)
        if done:
            success = bool(info.get('success', False))
        mask.fill_(0.0 if done else 1.0)
        actor_state = n_as * mask
        critic_state = n_cs * mask
        if done: break
    rets.append(ret); succ.append(float(success))
    if ret > best[0]: best = (ret, seed)
print(f'success_rate={np.mean(succ):.3f}  avg_return={np.mean(rets):.2f}  best_seed={best[1]} best_return={best[0]:.2f}')

# Render best episode
best_seed = best[1]
env = make_eval_env(best_seed)
obs, _ = env.reset(seed=best_seed)
actor_state, critic_state = policy.initial_states(1, device)
mask = torch.zeros(1, 1, device=device)
frames, traj_x, traj_y, cast_x, cast_y = [], [], [], [], []
for t in range(env.max_steps):
    obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
    with torch.no_grad():
        a, _, _, n_as, n_cs = policy.act(obs_t, actor_state, critic_state, mask, deterministic=True)
    traj_x.append(env.x_mm); traj_y.append(env.y_mm)
    obs, _, term, trunc, info = env.step(a.squeeze(0).cpu().numpy())
    if info.get('event_is_high_cast_like'):
        cast_x.append(env.x_mm); cast_y.append(env.y_mm)
    frames.append(render_rollout_frame(env, traj_x, traj_y, cast_x, cast_y, t,
                                       title=f'PPO seed={best_seed} step={t}'))
    done = bool(term or trunc)
    mask.fill_(0.0 if done else 1.0)
    actor_state = n_as * mask; critic_state = n_cs * mask
    if done: break

gif_path = os.path.join(RUN_DIR, 'plots', 'best_agent.gif')
save_gif(frames, gif_path, fps=15)
display(DisplayImage(data=open(gif_path, 'rb').read(), format='gif'))